In [1]:
!sudo apt update
!sudo apt install -y zstd
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,436 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

In [2]:

import subprocess

# Start Ollama server in the background
# The Popen call ensures the server runs in the background and doesn't block the cell execution.
# stdout=subprocess.DEVNULL and stderr=subprocess.DEVNULL suppress output from the ollama server itself
# while it's starting, as it can be quite verbose.
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Ollama server started in the background.")

Ollama server started in the background.
Ollama server started in the background.


In [3]:

import time

# Give Ollama a moment to fully initialize. Adjust sleep time if needed.
time.sleep(5)

print("Attempting to pull model...")
!ollama pull llama3

Attempting to pull model...



In [4]:
!pip install langchain-ollama

In [5]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
from langchain_ollama import ChatOllama

# Initialize the model with your exact version
llm = ChatOllama(
    model="llama3",
    temperature=0,
    device=device
)

In [7]:
llm.invoke("what do llamas eat?").content

"Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet typically consists of:\n\n1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.\n2. Hay: High-quality hay, such as timothy or alfalfa, is a staple in many llama diets. Hay provides essential fiber and nutrients.\n3. Grains: Some llamas may receive grains like oats, barley, or corn as part of their diet. However, it's essential to provide these in moderation, as excessive grain consumption can lead to health issues.\n4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.\n5. Minerals: Llamas require access to mineral supplements, which provide essential nutrients like calcium, phosphorus, and salt.\n\nIn the wild, llamas would typically roam in herds and forage on a variety of plants, including:\n\n* Grasses (e.g., bunchgrass, wheat gr

# Session Function

In [9]:
import json
from tqdm import tqdm
file_path = "instruction-data-with-response.json"
with open(file_path, "r") as file:
  test_data = json.load(file)
def format_input(entry):
  instruction_text = (
  f"Below is an instruction that describes a task. "
  f"Write a response that appropriately completes the request."
  f"\n\\n### Instruction:\\n{entry['instruction']}"
  )
  input_text = (
  f"\n\\n### Input:\\n{entry['input']}" if entry["input"] else ""
  )
  return instruction_text + input_text

In [10]:
test_data[0]

{'instruction': 'Rewrite the sentence using a simile.',
 'input': 'The car is very fast.',
 'output': 'The car is as fast as lightning.',
 'model_response': 'The car is as fast as a cheetah.'}

In [12]:

for entry in test_data[:3]:
  prompt = (
      f"Given the input {format_input(entry)} "
      f"and correct output {entry["output"]} "
      f"score the model response {entry["model_response"]} "
      f"on a scale from 0 to 100, where 100 is the best score"
      f"Just answer with the score."
  )

  print("\nDataset response:")
  print(">>", entry['output'])
  print("\nModel response:")
  print(">>", entry["model_response"])
  print("\nScore:")
  print(">>", llm.invoke(prompt).content)
  print("\n-------------------------")



Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a cheetah.

Score:
>> 80

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> The type of cloud typically associated with thunderstorms is a cumulus clouds type.

Score:
>> 44

-------------------------

Dataset response:
>> Jane Austen.

Model response:
>> The author of 'Pride and Prejudice' is George Orwell.

Score:
>> 4

-------------------------


# Evaluating the whole test_data

In [13]:
def generate_model_scores(json_data, json_key, model="llama3"):
  scores = []
  for entry in tqdm(json_data, desc="Scoring entries"):
    prompt = (
    f"Given the input `{format_input(entry)}` "
    f"and correct output `{entry['output']}`, "
    f"score the model response `{entry[json_key]}`"
    f" on a scale from 0 to 100, where 100 is the best score. "
    f"Respond with the integer number only." #1
      )
    score = llm.invoke(prompt).content
    try:
      scores.append(int(score))
    except ValueError:
      print(f"Could not convert score: {score}")
      continue
  return scores

In [14]:
scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [00:39<00:00,  2.81it/s]

Number of scores: 110 of 110
Average score: 35.77

